# Mejoras IES Triage — Guía de Examen

## Tipo de problema
**Agente de clasificación de correos** con RAG (Retrieval-Augmented Generation).
El agente recibe correos, busca respuestas en una base de conocimiento (FAQs) y decide:
- Responder automáticamente (si la FAQ cubre la pregunta)
- Escalar a atención al cliente (si no hay respuesta en la base de conocimiento)

## Arquitectura RAG
```
Correo entrante
    ↓
[Embedding del correo]
    ↓
[Búsqueda en ChromaDB de FAQs similares]
    ↓
[LLM: ¿puedo responder con estas FAQs?]
    ↓
Responder CSV  ←  Sí  /  No  →  Escalar CSV
```

## Mejoras implementadas
1. **Procesar múltiples JSONs** de una carpeta `inbox/` (no solo un archivo fijo)
2. **Evitar duplicados** con registro de IDs procesados
3. **Manejo de fallos** por JSON corrupto o campos faltantes
4. **RAG 2.0 conceptos** (solo viables si hay modelos descargados)


## Paso 1 — Procesar múltiples JSONs de inbox/

**Por qué es mejor que un JSON fijo?**
El proyecto original procesa un único archivo JSON hardcodeado.
En el desafío real, los correos llegan en lotes. Una carpeta `inbox/` es más realista:
puedes añadir un JSON nuevo y se procesa en la siguiente ejecución.

**Por qué mover a `processed/` después de leer?**
Si el script falla a mitad del procesamiento, no queremos volver a procesar
los correos ya respondidos. Mover el archivo confirma que fue procesado.

**Por qué una carpeta `failed/` separada?**
Si un JSON está corrupto, moverlo a `failed/` permite inspeccionarlo después
sin perderlo ni reprocesarlo infinitamente.


In [ ]:
from pathlib import Path
import json
import shutil

INBOX_DIR    = Path("inbox")
PROCESSED_DIR = Path("processed")
FAILED_DIR   = Path("failed")

# Crear directorios si no existen
for directory in [INBOX_DIR, PROCESSED_DIR, FAILED_DIR]:
    directory.mkdir(exist_ok=True)

print("Directorios creados:", INBOX_DIR, PROCESSED_DIR, FAILED_DIR)


In [ ]:
def cargar_correos_de_json(json_path: Path) -> list[dict]:
    """
    Carga correos desde un archivo JSON.
    Acepta tanto una lista de correos como un objeto con clave "correos".
    Retorna lista vacía si el archivo está corrupto.
    """
    try:
        data = json.loads(json_path.read_text(encoding="utf-8"))
        # El JSON puede ser una lista directa o {"correos": [...]}
        if isinstance(data, list):
            return data
        elif isinstance(data, dict) and "correos" in data:
            return data["correos"]
        else:
            print(f"Formato inesperado en {json_path}: {type(data)}")
            return []
    except json.JSONDecodeError as e:
        print(f"JSON corrupto en {json_path}: {e}")
        return []

def procesar_todos_los_json() -> list[dict]:
    """
    Lee todos los JSON de inbox/, mueve los procesados a processed/
    y los corruptos a failed/.
    """
    todos_los_correos = []
    for json_path in sorted(INBOX_DIR.glob("*.json")):
        correos = cargar_correos_de_json(json_path)
        if correos:
            todos_los_correos.extend(correos)
            shutil.move(str(json_path), str(PROCESSED_DIR / json_path.name))
            print(f"Procesado: {json_path.name} ({len(correos)} correos)")
        else:
            shutil.move(str(json_path), str(FAILED_DIR / json_path.name))
            print(f"Fallido: {json_path.name} (JSON vacío o corrupto)")
    return todos_los_correos

# Prueba: crear un JSON de ejemplo y procesarlo
test_json = INBOX_DIR / "lote_001.json"
test_json.write_text(json.dumps([
    {"id": "C001", "asunto": "Horario de atención", "mensaje": "¿A qué hora abren?"},
    {"id": "C002", "asunto": "Devolución", "mensaje": "Quiero devolver un producto roto"},
], ensure_ascii=False, indent=2), encoding="utf-8")

correos = procesar_todos_los_json()
print(f"\nTotal correos cargados: {len(correos)}")


## Paso 2 — Evitar duplicados con registro de IDs

**Por qué podría haber duplicados?**
Si el script se interrumpe justo antes de mover el JSON a `processed/`,
el mismo JSON se procesará de nuevo en la siguiente ejecución, respondiendo dos veces al mismo correo.

**Solución**: guardar los IDs de correos ya procesados en un archivo de texto.
Antes de procesar cada correo, comprobar si su ID ya está en esa lista.


In [ ]:
PROCESSED_IDS_FILE = Path("processed_ids.txt")

def leer_ids_procesados() -> set:
    "Carga los IDs de correos ya procesados desde el archivo de registro."
    if not PROCESSED_IDS_FILE.exists():
        return set()
    return set(PROCESSED_IDS_FILE.read_text(encoding="utf-8").splitlines())

def registrar_id_procesado(correo_id: str) -> None:
    "Añade un ID al registro de procesados (modo append, un ID por línea)."
    with PROCESSED_IDS_FILE.open("a", encoding="utf-8") as f:
        f.write(str(correo_id) + "\n")

# Uso en el bucle principal:
ids_ya_procesados = leer_ids_procesados()

for correo in correos:
    correo_id = correo.get("id")
    if correo_id in ids_ya_procesados:
        print(f"Saltando {correo_id}: ya procesado")
        continue
    # ... procesar correo ...
    print(f"Procesando: {correo_id} — {correo.get('asunto', 'sin asunto')}")
    registrar_id_procesado(correo_id)
    ids_ya_procesados.add(correo_id)


## Paso 3 — Manejo de campos faltantes en correos

**Por qué validar los campos?**
Un correo corrupto o mal formado (sin 'id', sin 'mensaje') puede hacer que el agente
crashee en medio del procesamiento, dejando los siguientes correos sin procesar.

**Estrategia**: validar al principio y saltar el correo si falta información esencial.
Un correo sin mensaje no se puede procesar. Un correo sin ID no se puede registrar.


In [ ]:
CAMPOS_REQUERIDOS = ["id", "asunto", "mensaje"]

def validar_correo(correo: dict) -> tuple:
    """
    Comprueba que el correo tiene todos los campos necesarios.
    Retorna (True, "") si es válido, (False, motivo) si no.
    """
    for campo in CAMPOS_REQUERIDOS:
        if campo not in correo:
            return False, f"Falta campo: {campo}"
        if not str(correo[campo]).strip():
            return False, f"Campo vacío: {campo}"
    return True, ""

# Ejemplo de uso:
correos_test = [
    {"id": "C001", "asunto": "Horario", "mensaje": "¿Cuándo abren?"},
    {"id": "C002", "asunto": "Roto"},         # falta "mensaje"
    {"asunto": "Consulta", "mensaje": "..."},  # falta "id"
]

for correo in correos_test:
    valido, motivo = validar_correo(correo)
    estado = "OK" if valido else f"SALTADO: {motivo}"
    print(f"  Correo {correo.get('id', 'sin-id')}: {estado}")


## Conceptos RAG 2.0 (solo si hay modelos descargados)

Estas mejoras **no son viables sin internet** salvo que los modelos estén cacheados.
Menciónalas en el examen como 'mejoras posibles' sin implementarlas si no tienes los modelos.

### Multi-query RAG
En lugar de buscar solo la pregunta original, generar 3 variaciones y buscar cada una:
- Original: '¿Cuánto tarda el envío?'
- Variación 1: '¿En cuántos días llega el pedido?'
- Variación 2: '¿Cuál es el plazo de entrega?'

Mejora el recall (menos casos donde el RAG no encuentra nada relevante).

### Routing
Antes de buscar en ChromaDB, clasificar el correo:
- Saludo / agradecimiento → responder directamente sin RAG
- Pregunta de FAQ → RAG normal
- Queja / problema complejo → escalar directamente sin RAG

Ahorra llamadas al LLM y al embedding model.


## Explicación para el examen

> *'Las mejoras al agente de triage son de robustez operacional:
> en vez de procesar un JSON fijo, leemos todos los archivos de una carpeta inbox/
> y los movemos a processed/ o failed/ después de procesarlos.
> Usamos un registro de IDs para evitar responder dos veces al mismo correo si el script
> se interrumpe. Validamos los campos requeridos antes de procesar para no crashear
> con datos malformados.'*

## Problemas frecuentes

| Problema | Solución |
|----------|----------|
| ChromaDB no arranca | Borrar `chroma_db/` y reiniciar |
| JSON procesado dos veces | Añadir registro de IDs |
| Correo sin campo 'id' causa crash | Validar antes de procesar |
| LLM no disponible sin internet | Usar Ollama local (ollama pull llama3.2) |
| Embeddings lentos | Usar modelo pequeño: `all-MiniLM-L6-v2` |
